In [14]:
import pandas as pd
import bonesis

In [22]:
regulations_path = "./hematopoiesis_influences.csv"
cell_states_path = "../grn/boolean_grn_matrix.csv"
genes = ["Egr1","Junb","Bclaf1","Myc","Fli1","Gata2","Spi1","Cebpa","Gata1","Klf1","Tal1","Ikzf1","Zfpm1"]

In [16]:
# ---------------------------------------------------------
# 1. Load and parse the Gene Regulation Network (PKN)
# ---------------------------------------------------------
print("Loading regulations...")
df_reg = pd.read_csv(regulations_path, sep="\t")

# Drop rows missing essential interaction data
df_reg = df_reg.dropna(subset=['tf', 'mor', 'target'])

influences = []
for _, row in df_reg.iterrows():
    source = str(row['tf'])
    target = str(row['target'])
    try:
        sign = int(row['mor'])
        if sign in [1, -1]:
            # BoNesis expects a list of tuples: (source, target, {"sign": 1 or -1})
            influences.append((source, target, {"sign": sign}))
    except ValueError:
        continue

# Instantiate the influence graph.
# 'maxclause' limits the number of clauses in the disjunctive normal form (DNF) 
# of the local update rules (e.g., max 3 regulators grouped in an OR clause).
dom = bonesis.InfluenceGraph(influences, maxclause=3)

Loading regulations...


In [17]:
dom

# computing graph layout...


In [18]:
# ---------------------------------------------------------
# 2. Load and parse the discretized Cell States
# ---------------------------------------------------------
print("Loading cell states...")
df_states = pd.read_csv(cell_states_path)

data = {}
gene_columns = [col for col in df_states.columns if col != 'Cell_Type']

for _, row in df_states.iterrows():
    cell_type = str(row['Cell_Type'])
    cell_data = {}
    for gene in gene_columns:
        val = row[gene]
        if pd.isna(val):
            continue
        val_str = str(val).strip()
        # If the value is a boolean state (0 or 1), add it.
        # '*' (wildcard) is ignored, leaving the gene unconstrained in this state.
        if val_str in ['0', '1', '0.0', '1.0']:
            cell_data[gene] = int(float(val_str))
    data[cell_type] = cell_data

Loading cell states...


In [19]:
for k,v in data.items():
    print(k, v)

iHSC {'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Ikzf1': 0, 'Zfpm1': 0}
ifnHSC {'Egr1': 0, 'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Zfpm1': 0}
pEr {'Egr1': 0, 'Junb': 0, 'Bclaf1': 1, 'Myc': 1, 'Cebpa': 0, 'Gata1': 1, 'Klf1': 1}
pLymph {'Junb': 1, 'Myc': 0, 'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Tal1': 0, 'Zfpm1': 0}
pMk {'Egr1': 0, 'Junb': 0, 'Bclaf1': 1, 'Fli1': 1, 'Gata2': 1, 'Cebpa': 0, 'Klf1': 0, 'Ikzf1': 0}
pNeuMast {'Egr1': 0, 'Spi1': 1, 'Gata1': 0, 'Klf1': 0, 'Tal1': 0, 'Zfpm1': 0}
preDiff {'Egr1': 0, 'Spi1': 1, 'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Zfpm1': 0}
qHSC {'Egr1': 1, 'Junb': 1, 'Fli1': 1, 'Gata2': 1, 'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Zfpm1': 0}
srHSC {'Egr1': 0, 'Cebpa': 0, 'Gata1': 0, 'Klf1': 0, 'Tal1': 0, 'Zfpm1': 0}


In [23]:
data["zero"] = { x: 0 for x in genes }

In [33]:
bo = bonesis.BoNesis(dom, data)

<img src="./hematopoiesis-trajectory.png" width="500"/>

In [34]:
fLymph = bo.fixed(~bo.obs("pLymph"))
fEr = bo.fixed(~bo.obs("pEr"))
fMk = bo.fixed(~bo.obs("pMk"))
fNeuMast = bo.fixed(~bo.obs("pNeuMast"))
start = ~bo.obs("iHSC")
# Reachability from iHSC
start >= ~bo.obs("srHSC")
start >= ~bo.obs("qHSC")
start >= fLymph
start >= ~bo.obs("preDiff") >= fEr
start >= ~bo.obs("preDiff") >= fMk
start >= ~bo.obs("preDiff") >= fNeuMast
# Reverse reachability
~bo.obs("srHSC") >= start
~bo.obs("qHSC") >= start
# Non-reachability
~bo.obs("preDiff") / ~bo.obs("qHSC")
~bo.obs("preDiff") / ~bo.obs("srHSC")
~bo.obs("preDiff") / start
~bo.obs('zero') / fNeuMast
~bo.obs('zero') / fMk
~bo.obs('zero') / fEr
~bo.obs('zero') / fLymph

# Fixpoint constraint from iHSC
# ~bo.obs("iHSC") >> "fixpoints" ^ {bo.obs(obs) for obs in ["pLymph", "pNeuMast","pEr","pMk","zero"]};

nonreach('<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x10ea498b0>', '<bonesis.language.ManagedIface.__init__.<locals>.managed.<locals>.Managed object at 0x10fd91a60>')

In [35]:
bo.boolean_networks(limit=1).count()

Grounding...done in 0.1s


1

In [36]:
for bn in bo.boolean_networks(limit=1):
    print(bn)

Grounding...done in 0.1s
Bclaf1 <- 1
Cebpa <- (!Gata1&Spi1)|(!Tal1&Spi1)|(!Zfpm1&Spi1)
Egr1 <- Junb&Gata2
Fli1 <- (!Klf1&Fli1&Gata1)|(!Klf1&Fli1&Gata2)|(!Klf1&Junb&Gata1&Gata2)
Gata1 <- !Ikzf1
Gata2 <- (!Zfpm1&!Spi1)|(Cebpa&!Gata1&Gata2)|(Egr1&Junb&!Gata1&!Spi1)
Ikzf1 <- !Gata1&Gata2
Junb <- (Egr1&Junb)|(Myc&Egr1)
Klf1 <- !Fli1
Myc <- Myc|(Cebpa&Spi1)|(Junb&Spi1)
Spi1 <- (Cebpa&!Gata1)|(Cebpa&!Gata2)
Tal1 <- 0
Zfpm1 <- 0



Discussion points: 

  - How "tight" discretization can we still use and get a satisfying model?
  - Will using "exact" graph make the inference harder or easier?
  - Try to disable the "all fixedpoints reachable from iHSC" constraint. How does it impact inference performance?

In [ ]:
projections = bo.local_functions()

for var in genes:
    with projections.view(var) as view:
        functions = [f for f in view]
        print(f"Variable `{var}` can have `{len(functions)}` update functions.")
        print(f"Showing first {min(len(functions), 10)}:")
        for f in functions[:10]:
            print("\t", str(f))
        

Grounding...done in 0.1s
Variable `Egr1` can have `2` update functions.
Showing first 2:
	 Junb&Gata2
	 (Egr1&Gata2)|(Junb&Gata2)
Variable `Junb` can have `6` update functions.
Showing first 6:
	 (Egr1&Junb)|(Myc&Egr1)
	 (Egr1&Junb)|(Myc&Junb)
	 Egr1&Junb
	 (Egr1&Junb)|(Myc&Egr1)|(Myc&Junb)
	 Egr1|(Myc&Junb)
	 Egr1
Variable `Bclaf1` can have `5` update functions.
Showing first 5:
	 Bclaf1|Myc
	 Bclaf1
	 Myc&Bclaf1
	 Myc
	 1
Variable `Myc` can have `688` update functions.
Showing first 10:
	 (Myc&Bclaf1)|(Cebpa&Junb&Spi1&Bclaf1)
	 (Myc&Bclaf1)|(Cebpa&Junb&Spi1)
	 (Myc&Bclaf1)|(Cebpa&Junb)
	 (Myc&Bclaf1)|(Cebpa&Spi1)
	 (Myc&Bclaf1)|(Cebpa&Spi1&Bclaf1)
	 (Myc&Bclaf1)|(Cebpa&Junb&Bclaf1)
	 (Myc&Bclaf1)|(Cebpa&Junb&Bclaf1)|(Junb&Myc&Cebpa)
	 (Myc&Bclaf1)|(Cebpa&Junb&Bclaf1)|(Junb&Myc&Cebpa&Spi1)
	 (Myc&Bclaf1)|(Cebpa&Junb&Bclaf1)|(Myc&Cebpa&Spi1)
	 (Myc&Bclaf1)|(Cebpa&Junb&Bclaf1)|(Cebpa&Junb&Spi1)
Variable `Fli1` can have `1165` update functions.
Showing first 10:
	 (!Klf1&Junb&Gata2)|(Jun